## Text Dataset (Already Done)

In [1]:
from recap import TextDataset
from torch.utils.data import DataLoader
import tiktoken

text_file = '../2 Working With Text Data/the-verdict.txt'
tokenizer = tiktoken.get_encoding("gpt2")
with open(text_file, "r") as f:
    text = f.read()

train_test_split = 0.7
train_text = text[:int(len(text) * train_test_split)]
test_text = text[int(len(text) * train_test_split):]

batch_size = 4
context_length = 256
train_dataset = TextDataset(train_text, tokenizer, context_length)
test_dataset = TextDataset(test_text, tokenizer, context_length)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    drop_last=True
)

## GPT Pre-Training Pipeline

In [2]:
import torch.nn as nn
import torch
from recap import GPT_Model


vocab_size = len(tokenizer._mergeable_ranks)
context_size = 256
emb_dim = 512
hidden_dim = emb_dim
dropout = 0.2
head_nums = 8
num_layers = 2
casual = True

lr = 3e-4
model = GPT_Model(hidden_dim, vocab_size, context_length, dropout, head_nums, num_layers)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

epochs = 5
eval_interval = 2
casual = True
epoch_list = []
train_losses = []
test_losses = []

In [5]:
for epoch in range(epochs):

    model.train()
    train_loss = 0

    for inputs, targets in train_dataloader:

        optimizer.zero_grad()
        preds = model(inputs, casual)
        loss = loss_fn(preds.view(-1, preds.size(-1)), targets.view(-1))
        loss.backward()
        train_loss += loss.item()
        optimizer.step()

    if epoch % eval_interval == 0 or epochs - epoch == 1:

        model.eval()
        test_loss = 0
        
        with torch.no_grad():
            for inputs, targets in test_dataloader:

                preds = model(inputs, casual)
                loss = loss_fn(preds.view(-1, preds.size(-1)), targets.view(-1))
                test_loss += loss.item()

        train_loss = train_loss / len(train_dataloader)
        test_loss = test_loss / len(test_dataloader)

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Eval Loss: {test_loss:.4f}"
        )
        
        epoch_list.append(epoch)
        train_losses.append(train_loss)
        test_losses.append(test_loss)

        break

Epoch 1/5 | Train Loss: 6.5738 | Eval Loss: 7.2170


## GPT Generation

In [6]:
def generate(model, text, tokenizer, context_length, max_length, casual, eot_token, temperature, top_k, return_text):

    token_ids = torch.tensor(tokenizer.encode(text)).unsqueeze(0)
    model.eval()

    for _ in range(max_length - len(token_ids[0])):

        with torch.no_grad():
            logits = model(token_ids[:, -context_length:], casual=casual)[:, -1, :]

            _, top_k_pos = torch.topk(logits, top_k)
            mask = torch.ones_like(logits, dtype=torch.bool)
            mask.scatter_(dim=-1, index=top_k_pos, value=False)
            logits = logits.masked_fill(mask, -torch.inf)

            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            id_next = torch.multinomial(probs, 1)

            if id_next.item() == eot_token:
                break

            token_ids = torch.cat((token_ids, id_next), dim=1)

    if return_text:
        return tokenizer.decode(token_ids.squeeze(0).tolist())

In [8]:
text = "What a sad day for "
max_length = 50
eot_token = None
temperature = 1.5
top_k = 16
return_text = True

generated = generate(model, text, tokenizer,  context_length, max_length, casual, eot_token, temperature, top_k, return_text)
generated

"What a sad day for -- was, a to he.'s, I, in that the the was., it to the a to to the that I the it, had it he of the her--, to a-- his that the"

## Dataclass + Config Refactor (Same Style as Your Code)

This version keeps your same naming and flow (`casual`, `generate(...)`, same training loop style),
but wraps parameters into config dataclasses so it is easier to reuse in a `.py` script.


In [ ]:
from dataclasses import dataclass
from typing import Optional

import torch.nn as nn
import torch
from torch.utils.data import DataLoader
import tiktoken

from recap import TextDataset, GPT_Model


@dataclass
class DataConfig:
    text_file: str = '../2 Working With Text Data/the-verdict.txt'
    train_test_split: float = 0.7
    batch_size: int = 4
    context_length: int = 256


@dataclass
class ModelConfig:
    context_length: int = 256
    emb_dim: int = 512
    hidden_dim: int = 512
    dropout: float = 0.2
    head_nums: int = 8
    num_layers: int = 2
    casual: bool = True


@dataclass
class TrainConfig:
    lr: float = 3e-4
    epochs: int = 5
    eval_interval: int = 2


@dataclass
class GenerationConfig:
    text: str = 'What a sad day for '
    max_length: int = 50
    eot_token: Optional[int] = None
    temperature: float = 1.5
    top_k: int = 16
    return_text: bool = True


def load_data(data_cfg: DataConfig):

    tokenizer = tiktoken.get_encoding('gpt2')

    with open(data_cfg.text_file, 'r') as f:
        text = f.read()

    train_text = text[:int(len(text) * data_cfg.train_test_split)]
    test_text = text[int(len(text) * data_cfg.train_test_split):]

    train_dataset = TextDataset(train_text, tokenizer, data_cfg.context_length)
    test_dataset = TextDataset(test_text, tokenizer, data_cfg.context_length)

    train_dataloader = DataLoader(
        train_dataset,
        batch_size=data_cfg.batch_size,
        shuffle=True,
        drop_last=True,
    )

    test_dataloader = DataLoader(
        test_dataset,
        batch_size=data_cfg.batch_size,
        shuffle=False,
        drop_last=True,
    )

    return tokenizer, train_dataloader, test_dataloader


def generate(model, text, tokenizer, context_length, max_length, casual, eot_token, temperature, top_k, return_text):

    token_ids = torch.tensor(tokenizer.encode(text)).unsqueeze(0)
    model.eval()

    for _ in range(max_length - len(token_ids[0])):

        with torch.no_grad():
            logits = model(token_ids[:, -context_length:], casual=casual)[:, -1, :]

            top_k = min(top_k, logits.size(-1))
            _, top_k_pos = torch.topk(logits, top_k)
            mask = torch.ones_like(logits, dtype=torch.bool)
            mask.scatter_(dim=-1, index=top_k_pos, value=False)
            logits = logits.masked_fill(mask, -torch.inf)

            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            id_next = torch.multinomial(probs, 1)

            if eot_token is not None and id_next.item() == eot_token:
                break

            token_ids = torch.cat((token_ids, id_next), dim=1)

    if return_text:
        return tokenizer.decode(token_ids.squeeze(0).tolist())

    return token_ids


class GPTPretrainer:

    def __init__(self, tokenizer, model_cfg: ModelConfig, train_cfg: TrainConfig):

        self.tokenizer = tokenizer
        self.model_cfg = model_cfg
        self.train_cfg = train_cfg

        vocab_size = len(self.tokenizer._mergeable_ranks)

        self.model = GPT_Model(
            self.model_cfg.hidden_dim,
            vocab_size,
            self.model_cfg.context_length,
            self.model_cfg.dropout,
            self.model_cfg.head_nums,
            self.model_cfg.num_layers,
        )

        self.loss_fn = nn.CrossEntropyLoss()
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.train_cfg.lr)

        self.epoch_list = []
        self.train_losses = []
        self.test_losses = []


    def fit(self, train_dataloader, test_dataloader):

        for epoch in range(self.train_cfg.epochs):

            self.model.train()
            train_loss = 0

            for inputs, targets in train_dataloader:

                self.optimizer.zero_grad()
                preds = self.model(inputs, self.model_cfg.casual)
                loss = self.loss_fn(preds.view(-1, preds.size(-1)), targets.view(-1))
                loss.backward()
                train_loss += loss.item()
                self.optimizer.step()

            if epoch % self.train_cfg.eval_interval == 0 or self.train_cfg.epochs - epoch == 1:

                self.model.eval()
                test_loss = 0

                with torch.no_grad():
                    for inputs, targets in test_dataloader:

                        preds = self.model(inputs, self.model_cfg.casual)
                        loss = self.loss_fn(preds.view(-1, preds.size(-1)), targets.view(-1))
                        test_loss += loss.item()

                train_loss = train_loss / len(train_dataloader)
                test_loss = test_loss / len(test_dataloader)

                print(
                    f'Epoch {epoch + 1}/{self.train_cfg.epochs} | '
                    f'Train Loss: {train_loss:.4f} | '
                    f'Eval Loss: {test_loss:.4f}'
                )

                self.epoch_list.append(epoch)
                self.train_losses.append(train_loss)
                self.test_losses.append(test_loss)

        return self.epoch_list, self.train_losses, self.test_losses